In [1]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

Forward propagation for one epoch

In [3]:

import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# 1. Use CPU
device = torch.device("cpu")
print("Device:", device)

# 2. Load prepared dataset
X = np.load(r"C:\Users\lenovo\Desktop\Sathyaraj\sathya\data\output\X.npy")
y = np.load(r"C:\Users\lenovo\Desktop\Sathyaraj\sathya\data\output\y.npy")

print("X shape:", X.shape)
print("y shape:", y.shape)

# 3. Convert to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# 4. Create batches
dataset = TensorDataset(X_tensor, y_tensor)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=False
)

# 5. Define the CNN
class BearingCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 6. Initialize model with initial weights
model = BearingCNN().to(device)

# Evaluation mode disables dropout for this demonstration
model.eval()

# 7. Run one forward-only epoch
all_outputs = []
all_predictions = []

with torch.no_grad():
    for batch_number, (inputs, targets) in enumerate(loader, start=1):
        inputs = inputs.to(device)

        # Forward propagation
        outputs = model(inputs)

        # Choose class with the highest logit
        predictions = outputs.argmax(dim=1)

        all_outputs.append(outputs)
        all_predictions.append(predictions)

        if batch_number == 1:
            print("\nFirst batch input shape:", inputs.shape)
            print("First batch output shape:", outputs.shape)
            print("First 5 logits:\n", outputs[:5])
            print("First 5 predictions:", predictions[:5].tolist())

# 8. Combine batch results
all_outputs = torch.cat(all_outputs, dim=0)
all_predictions = torch.cat(all_predictions, dim=0)

print("\nForward-only epoch complete!")
print("Total windows processed:", len(all_predictions))
print("All output logits shape:", all_outputs.shape)
print("All predictions shape:", all_predictions.shape)
print("No weights were updated.")


Device: cpu
X shape: (5926, 1, 1024)
y shape: (5926,)

First batch input shape: torch.Size([64, 1, 1024])
First batch output shape: torch.Size([64, 4])
First 5 logits:
 tensor([[-0.0729, -0.1166, -0.0084,  0.0046],
        [-0.0727, -0.1164, -0.0080,  0.0053],
        [-0.0727, -0.1163, -0.0090,  0.0049],
        [-0.0724, -0.1157, -0.0079,  0.0053],
        [-0.0734, -0.1175, -0.0086,  0.0043]])
First 5 predictions: [3, 3, 3, 3, 3]

Forward-only epoch complete!
Total windows processed: 5926
All output logits shape: torch.Size([5926, 4])
All predictions shape: torch.Size([5926])
No weights were updated.


In [10]:
import numpy as np

groups = np.load(r"C:\Users\lenovo\Desktop\Sathyaraj\sathya\data\output\groups.npy")

print("Groups defined successfully!")
print("Total windows:", len(groups))
print("Unique recordings:", len(np.unique(groups)))
print("First 10 groups:", groups[:10])

assert len(groups) == len(X) == len(y), \
    "Mismatch between X, y, and groups!"

Groups defined successfully!
Total windows: 5926
Unique recordings: 40
First 10 groups: ['B007_0.mat' 'B007_0.mat' 'B007_0.mat' 'B007_0.mat' 'B007_0.mat'
 'B007_0.mat' 'B007_0.mat' 'B007_0.mat' 'B007_0.mat' 'B007_0.mat']


In [14]:

# ==========================================
# 2. CLASS-WISE RECORDING-LEVEL SPLIT
# ==========================================

import numpy as np

rng = np.random.default_rng(42)

unique_groups = np.unique(groups)

# Determine the class belonging to each recording.
group_to_label = {}

for group in unique_groups:
    labels_for_group = np.unique(y[groups == group])

    if len(labels_for_group) != 1:
        raise ValueError(
            f"Recording {group} has multiple labels: {labels_for_group}"
        )

    group_to_label[group] = int(labels_for_group[0])

train_groups = []
val_groups = []
test_groups = []

class_names = {
    0: "Normal",
    1: "Inner-race fault",
    2: "Outer-race fault",
    3: "Ball fault"
}

# Split each class independently, keeping entire recordings together.
for class_id, class_name in class_names.items():

    class_groups = np.array([
        group for group in unique_groups
        if group_to_label[group] == class_id
    ])

    rng.shuffle(class_groups)
    n = len(class_groups)

    print(f"{class_name}: {n} unique recordings")

    if n < 3:
        raise ValueError(
            f"{class_name} has only {n} unique recordings. "
            "At least 3 are needed to put this class into all "
            "three splits without recording leakage."
        )

    n_test = max(1, round(n * 0.15))
    n_val = max(1, round(n * 0.15))

    # Always leave at least one recording for training.
    while n_test + n_val >= n:
        if n_val > 1:
            n_val -= 1
        elif n_test > 1:
            n_test -= 1
        else:
            raise ValueError(
                f"Cannot split {class_name} into three nonempty sets."
            )

    test_groups.extend(class_groups[:n_test])
    val_groups.extend(class_groups[n_test:n_test + n_val])
    train_groups.extend(class_groups[n_test + n_val:])

train_groups = np.array(train_groups)
val_groups = np.array(val_groups)
test_groups = np.array(test_groups)

# Convert recording assignments into window indices.
train_idx = np.flatnonzero(np.isin(groups, train_groups))
val_idx = np.flatnonzero(np.isin(groups, val_groups))
test_idx = np.flatnonzero(np.isin(groups, test_groups))

# Arrays used by the existing DataLoader section.
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

# Verify there is no overlap between recordings.
assert set(groups[train_idx]).isdisjoint(set(groups[val_idx]))
assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))
assert set(groups[val_idx]).isdisjoint(set(groups[test_idx]))

# Verify all classes appear in every split.
for name, indices in [
    ("Train", train_idx),
    ("Validation", val_idx),
    ("Test", test_idx)
]:
    labels = y[indices]

    print(f"\n{name}")
    print("Windows:", len(indices))
    print("Recordings:", len(np.unique(groups[indices])))
    print(
        "Class counts:",
        {
            class_id: int(np.sum(labels == class_id))
            for class_id in range(4)
        }
    )

    assert set(np.unique(labels)) == {0, 1, 2, 3}, (
        f"{name} is missing a class."
    )

print("\nAll splits contain all four classes.")
print("No recording leakage detected.")


Normal: 4 unique recordings
Inner-race fault: 12 unique recordings
Outer-race fault: 12 unique recordings
Ball fault: 12 unique recordings

Train
Windows: 3558
Recordings: 26
Class counts: {0: 710, 1: 948, 2: 952, 3: 948}

Validation
Windows: 1184
Recordings: 7
Class counts: {0: 472, 1: 238, 2: 236, 3: 238}

Test
Windows: 1184
Recordings: 7
Class counts: {0: 474, 1: 236, 2: 237, 3: 237}

All splits contain all four classes.
No recording leakage detected.


activation fn = relu ;
optimiser = adam

In [17]:

# ============================================================
# CNN-BASED FAULT DETECTION FOR SPACE ROVERS AND MACHINERY
# Complete Training Pipeline
# ============================================================

import copy
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 0.001
PATIENCE = 7

device = torch.device("cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

CLASS_NAMES = {
    0: "Normal",
    1: "Inner-race fault",
    2: "Outer-race fault",
    3: "Ball fault"
}

print("Device:", device)
print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)
print("Maximum epochs:", EPOCHS)

# ============================================================
# 2. LOAD PREPROCESSED DATA
# ============================================================

X = np.load(r"C:\Users\lenovo\Desktop\Sathyaraj\sathya\data\output\X.npy")
y = np.load(r"C:\Users\lenovo\Desktop\Sathyaraj\sathya\data\output\y.npy")
groups = np.load(r"C:\Users\lenovo\Desktop\Sathyaraj\sathya\data\output\groups.npy")

X = X.astype(np.float32)
y = y.astype(np.int64)
groups = groups.astype(str)

assert len(X) == len(y) == len(groups), (
    "X, y, and groups must contain the same number of windows."
)

assert X.ndim == 3 and X.shape[1] == 1, (
    "Expected X shape: (number_of_windows, 1, window_size)."
)

print("\nDataset loaded")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique recordings:", len(np.unique(groups)))

# ============================================================
# 3. RECORDING-LEVEL SPLIT
#    Windows from one recording cannot cross splits.
# ============================================================

rng = np.random.default_rng(SEED)
unique_groups = np.unique(groups)

group_to_label = {}

for group in unique_groups:
    labels_for_group = np.unique(y[groups == group])

    if len(labels_for_group) != 1:
        raise ValueError(
            f"Recording {group} has multiple labels: {labels_for_group}"
        )

    group_to_label[group] = int(labels_for_group[0])

train_groups = []
val_groups = []
test_groups = []

print("\nUnique recordings per class:")

for class_id, class_name in CLASS_NAMES.items():

    class_groups = np.array([
        group for group in unique_groups
        if group_to_label[group] == class_id
    ])

    rng.shuffle(class_groups)
    n = len(class_groups)

    print(f"{class_name}: {n}")

    if n < 3:
        raise ValueError(
            f"{class_name} has only {n} unique recordings. "
            "At least 3 separate recordings are required for "
            "training, validation, and testing."
        )

    # Reserve approximately 15% of recordings for validation
    # and 15% for testing, separately within each class.
    n_test = max(1, round(n * 0.15))
    n_val = max(1, round(n * 0.15))

    # Leave at least one recording for training.
    while n_test + n_val >= n:
        if n_val > 1:
            n_val -= 1
        elif n_test > 1:
            n_test -= 1
        else:
            raise ValueError(
                f"Cannot split class {class_name} into three sets."
            )

    test_groups.extend(class_groups[:n_test])
    val_groups.extend(class_groups[n_test:n_test + n_val])
    train_groups.extend(class_groups[n_test + n_val:])

train_groups = np.array(train_groups)
val_groups = np.array(val_groups)
test_groups = np.array(test_groups)

train_idx = np.flatnonzero(np.isin(groups, train_groups))
val_idx = np.flatnonzero(np.isin(groups, val_groups))
test_idx = np.flatnonzero(np.isin(groups, test_groups))

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

# Check recording separation.
assert set(groups[train_idx]).isdisjoint(set(groups[val_idx]))
assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))
assert set(groups[val_idx]).isdisjoint(set(groups[test_idx]))

print("\nSplit summary:")

for name, indices in [
    ("Training", train_idx),
    ("Validation", val_idx),
    ("Testing", test_idx)
]:
    labels = y[indices]

    print(f"\n{name}:")
    print("Windows:", len(indices))
    print("Recordings:", len(np.unique(groups[indices])))
    print(
        "Class counts:",
        {
            c: int(np.sum(labels == c))
            for c in CLASS_NAMES
        }
    )

    if set(np.unique(labels)) != set(CLASS_NAMES):
        raise ValueError(
            f"{name} does not contain all four classes. "
            "Check the dataset labels and recording counts."
        )

print("\nRecording-level split verified. No recording leakage.")

# ============================================================
# 4. CREATE PYTORCH DATA LOADERS
# ============================================================

train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train)
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(y_val)
)

test_dataset = TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(y_test)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ============================================================
# 5. DEFINE THE CNN MODEL
# ============================================================

class BearingCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = BearingCNN(num_classes=4).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("\nCNN initialized")
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

# ============================================================
# 6. TRAINING LOOP WITH EARLY STOPPING
# ============================================================

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

for epoch in range(EPOCHS):

    # ---------------- TRAINING ----------------
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * inputs.size(0)
        train_correct += (
            outputs.argmax(dim=1) == targets
        ).sum().item()
        train_total += targets.size(0)

    train_loss = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # ---------------- VALIDATION ----------------
    model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            val_loss_sum += loss.item() * inputs.size(0)
            val_correct += (
                outputs.argmax(dim=1) == targets
            ).sum().item()
            val_total += targets.size(0)

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    # ---------------- BEST CHECKPOINT ----------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2%} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2%}"
    )

    # ---------------- EARLY STOPPING ----------------
    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch + 1}.")
        break

# ============================================================
# 7. RESTORE BEST MODEL AND SAVE CHECKPOINT
# ============================================================

if best_model_state is None:
    raise RuntimeError("Training did not produce a model checkpoint.")

model.load_state_dict(best_model_state)
model.eval()

model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "bearing_cnn.pth"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_names": CLASS_NAMES,
        "window_size": X.shape[-1],
        "best_validation_loss": best_val_loss
    },
    model_path
)

print("\nTraining complete!")
print("Best validation loss:", round(best_val_loss, 4))
print("Saved model:", model_path)

# ============================================================
# 8. FINAL EVALUATION ON THE HELD-OUT TEST SET
# ============================================================

model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)

        outputs = model(inputs)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(targets.numpy())

test_accuracy = accuracy_score(all_targets, all_predictions)

print("\nTest accuracy:", f"{test_accuracy:.2%}")

print("\nClassification report:")
print(
    classification_report(
        all_targets,
        all_predictions,
        labels=[0, 1, 2, 3],
        target_names=[CLASS_NAMES[i] for i in range(4)],
        zero_division=0
    )
)

print("Confusion matrix:")
print(
    confusion_matrix(
        all_targets,
        all_predictions,
        labels=[0, 1, 2, 3]
    )
)


Device: cpu
Optimizer: Adam
Learning rate: 0.001
Maximum epochs: 30

Dataset loaded
X shape: (5926, 1, 1024)
y shape: (5926,)
Unique recordings: 40

Unique recordings per class:
Normal: 4
Inner-race fault: 12
Outer-race fault: 12
Ball fault: 12

Split summary:

Training:
Windows: 3558
Recordings: 26
Class counts: {0: 710, 1: 948, 2: 952, 3: 948}

Validation:
Windows: 1184
Recordings: 7
Class counts: {0: 472, 1: 238, 2: 236, 3: 238}

Testing:
Windows: 1184
Recordings: 7
Class counts: {0: 474, 1: 236, 2: 237, 3: 237}

Recording-level split verified. No recording leakage.

CNN initialized
Trainable parameters: 11140
Epoch 01/30 | Train Loss: 1.2535 | Train Acc: 37.77% | Val Loss: 0.7342 | Val Acc: 56.84%
Epoch 02/30 | Train Loss: 0.8172 | Train Acc: 60.65% | Val Loss: 0.5286 | Val Acc: 84.63%
Epoch 03/30 | Train Loss: 0.5308 | Train Acc: 80.16% | Val Loss: 0.3399 | Val Acc: 89.44%
Epoch 04/30 | Train Loss: 0.3304 | Train Acc: 88.00% | Val Loss: 0.2251 | Val Acc: 92.99%
Epoch 05/30 | Train